# Unsupervised Clustering for Running State Detection — v3

**Sites:** Vandolah (VAN1) + Harquahala (CT1-CT3, ST1-ST3)  
**Primary Metrics:** Precision, Recall, F1 (not accuracy)  
**Evaluation:** Site-agnostic — each method runs independently per site with identical hyperparameters  
**Hard constraint:** Every sample must receive an ON/OFF prediction — no data dropped or left unclassified  

### Key design decisions
1. **Metrics**: F1/Precision/Recall as primary (accuracy only as secondary reference)
2. **Preprocessing**: Documented pipeline — outlier clipping, RobustScaler, scale-invariant feature subsets
3. **100% coverage**: DBSCAN noise points assigned to nearest cluster centroid — mirrors production
4. **Site-agnostic**: Same method with identical config applied independently per site
5. **Ensemble methods**: Majority vote and weighted vote across base clusterers
6. **Production focus**: Identify the method maximizing worst-case F1 across all sites

---
## 1. Configuration & Data Loading

In [ ]:
import os, re, zipfile, json, io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import entropy, kurtosis, skew
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans, SpectralClustering, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.svm import OneClassSVM, SVC
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             accuracy_score, f1_score, precision_score, recall_score,
                             silhouette_score, confusion_matrix)
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from itertools import permutations
from collections import Counter

warnings.filterwarnings('ignore')
np.random.seed(42)

DATA_DIR = os.path.join(os.getcwd(), 'On_Off_data')
BAND_DEFS = [('B1',30000,300000),('B2',300000,3000000),
             ('B3',3000000,30000000),('B4',30000000,100000000)]
PATTERNS = [
    re.compile(r'^(Van\d+)_(\w+?)_(on|off)_(\d{4}_\d{2}_\d{2})\.zip$', re.I),
    re.compile(r'^(Harq\w+?)_(\w+?)_(on|off)_(\d{4}_\d{2}_\d{2})\.zip$', re.I),
]

def parse_filename(fname):
    for pat in PATTERNS:
        m = pat.match(fname)
        if m:
            unit = m.group(1).upper()
            site = 'Vandolah' if unit.startswith('VAN') else 'Harquahala'
            return unit, m.group(2), m.group(3).lower(), m.group(4).replace('_','-'), site
    return None

print(f'Data: {DATA_DIR}')

---
## 2. Feature Engineering

| Category | Features | Scale-Invariant? |
|----------|----------|------------------|
| Raw band powers | total_power, B1-B4_power, mean, max, std | No |
| Band fractions | B1-B4_frac (sum-to-1 per sample) | **Yes** |
| Log transforms | log1p of powers | Partially |
| Inter-band ratios | B2/B1, B3/B1, B4/B1, hi/lo, etc. | **Yes** |
| Spectral shape | centroid, spread, skew, kurtosis, entropy, flatness, slope | **Yes** |
| Peak features | peak_freq, peak_to_mean, crest_factor | **Yes** |

In [ ]:
def compute_features(freqs, powers):
    f = {}
    total = powers.sum()
    f['total_power'] = total
    f['mean_power'] = powers.mean()
    f['max_power'] = powers.max()
    f['std_power'] = powers.std()
    for bn, lo, hi in BAND_DEFS:
        mask = (freqs >= lo) & (freqs < hi)
        bp = powers[mask]; bsum = bp.sum()
        f[f'{bn}_power'] = bsum
        f[f'{bn}_mean'] = bp.mean() if len(bp) > 0 else 0
        f[f'{bn}_max'] = bp.max() if len(bp) > 0 else 0
        f[f'{bn}_std'] = bp.std() if len(bp) > 0 else 0
    for bn, _, _ in BAND_DEFS:
        f[f'{bn}_frac'] = f[f'{bn}_power'] / (total + 1e-10)
    f['log_total'] = np.log1p(total)
    for bn, _, _ in BAND_DEFS:
        f[f'{bn}_log'] = np.log1p(f[f'{bn}_power'])
    f['B2_B1_ratio'] = f['B2_power'] / (f['B1_power'] + 1e-6)
    f['B3_B1_ratio'] = f['B3_power'] / (f['B1_power'] + 1e-6)
    f['B4_B1_ratio'] = f['B4_power'] / (f['B1_power'] + 1e-6)
    f['B4_B2_ratio'] = f['B4_power'] / (f['B2_power'] + 1e-6)
    f['B3_B2_ratio'] = f['B3_power'] / (f['B2_power'] + 1e-6)
    f['hi_lo_ratio'] = (f['B3_power'] + f['B4_power']) / (f['B1_power'] + f['B2_power'] + 1e-6)
    f['log_B2_B1'] = np.log1p(f['B2_B1_ratio'])
    f['log_B4_B1'] = np.log1p(f['B4_B1_ratio'])
    f['log_hi_lo'] = np.log1p(f['hi_lo_ratio'])
    normp = powers / (total + 1e-10)
    freq_idx = np.arange(len(powers))
    centroid = np.sum(freq_idx * normp)
    spread = np.sqrt(np.sum((freq_idx - centroid)**2 * normp))
    f['spectral_centroid'] = centroid
    f['spectral_spread'] = spread
    f['spectral_skew'] = skew(powers)
    f['spectral_kurtosis'] = kurtosis(powers)
    normp_pos = np.clip(normp, 1e-20, None)
    f['spectral_entropy'] = entropy(normp_pos) / np.log(len(powers))
    log_mean = np.mean(np.log(powers + 1e-10))
    f['spectral_flatness'] = np.exp(log_mean) / (powers.mean() + 1e-10)
    peak_idx = np.argmax(powers)
    f['peak_freq_mhz'] = freqs[peak_idx] / 1e6
    f['peak_to_mean'] = powers[peak_idx] / (powers.mean() + 1e-10)
    log_p = np.log1p(powers)
    slope, _ = np.polyfit(freq_idx, log_p, 1)
    f['spectral_slope'] = slope
    cumsum = np.cumsum(powers)
    rolloff_idx = np.searchsorted(cumsum, 0.85 * total)
    f['rolloff_freq_mhz'] = freqs[min(rolloff_idx, len(freqs)-1)] / 1e6
    rms = np.sqrt(np.mean(powers**2))
    f['crest_factor'] = powers.max() / (rms + 1e-10)
    band_pows = [f[f'{bn}_power'] for bn, _, _ in BAND_DEFS]
    log_bands = np.log1p(band_pows)
    f['band_gradient'] = np.mean(np.diff(log_bands))
    f['band_range'] = log_bands.max() - log_bands.min()
    return f

print(f'Feature extractor: {len(compute_features(np.linspace(3e4,1e8,8000), np.random.rand(8000)))} features')

In [ ]:
records = []
for fname in sorted(os.listdir(DATA_DIR)):
    if not fname.endswith('.zip'): continue
    parsed = parse_filename(fname)
    if not parsed: continue
    unit, sensor, state, date_str, site = parsed
    with zipfile.ZipFile(os.path.join(DATA_DIR, fname)) as z:
        with z.open('ChartData.xlsx') as xf:
            df = pd.read_excel(io.BytesIO(xf.read()))
    freqs = pd.to_numeric(df.iloc[1:,0], errors='coerce')
    powers = pd.to_numeric(df.iloc[1:,1], errors='coerce')
    valid = freqs.notna() & powers.notna()
    feats = compute_features(freqs[valid].values, powers[valid].values)
    feats.update({'file': fname, 'unit': unit, 'sensor': sensor,
                  'state': state, 'date': date_str, 'site': site})
    records.append(feats)

data = pd.DataFrame(records)
data['running'] = (data['state'] == 'on').astype(int)
meta_cols = ['file','unit','sensor','state','date','site','running']
feature_cols = [c for c in data.columns if c not in meta_cols]
y_true = data['running'].values
sites = sorted(data['site'].unique())

SCALE_INVARIANT = [c for c in feature_cols if any(k in c for k in
    ['frac','ratio','hi_lo','spectral','entropy','flatness','slope',
     'rolloff','crest','peak_to_mean','peak_freq','gradient','range'])]

print(f'Loaded {len(data)} samples, {len(feature_cols)} features ({len(SCALE_INVARIANT)} scale-invariant)')
for sn in sites:
    mk = data['site']==sn
    print(f'  {sn}: {mk.sum()} ({data.loc[mk,"running"].sum()} ON / {(~data.loc[mk,"running"].astype(bool)).sum()} OFF)')

---
## 3. Preprocessing & Feature Subsets

In [ ]:
def preprocess(X_raw):
    """Pipeline: clip outliers at 1/99 percentile -> RobustScaler."""
    X = X_raw.copy().astype(float)
    for j in range(X.shape[1]):
        p01, p99 = np.percentile(X[:, j], [1, 99])
        X[:, j] = np.clip(X[:, j], p01, p99)
    return RobustScaler().fit_transform(X)

S1 = ['total_power']
S2 = ['total_power','B1_power','B2_power','B3_power','B4_power']
BAND_LOGS = ['B1_log','B2_log','B3_log','B4_log','log_total']
S3a = S2 + BAND_LOGS
S3b = S3a + [f'{b}_{s}' for b in ['B1','B2','B3','B4'] for s in ['mean','max','std']]
RATIOS = [c for c in feature_cols if 'ratio' in c or 'hi_lo' in c]
BAND_FRACS = ['B1_frac','B2_frac','B3_frac','B4_frac']
LOG_FEATS = [c for c in feature_cols if 'log' in c]
S3c = S3b + RATIOS + BAND_FRACS + [c for c in LOG_FEATS if c not in S3b]
SHAPE = [c for c in feature_cols if any(k in c for k in
    ['spectral','entropy','flatness','slope','rolloff','crest','peak_to_mean','peak_freq','gradient','range'])]
S3d = S3c + SHAPE
S3e = feature_cols
S_INV = SCALE_INVARIANT
S_INV_R = [c for c in S_INV if c not in SHAPE]

X_all = data[feature_cols].values.astype(float)
mi_scores = mutual_info_classif(X_all, y_true, random_state=42)
mi_rank = {c: i for i, (c,_) in enumerate(sorted(zip(feature_cols, mi_scores), key=lambda x: -x[1]))}
def cohens_d(col):
    on_v = data.loc[data['running']==1, col].values
    off_v = data.loc[data['running']==0, col].values
    ps = np.sqrt((on_v.std()**2 + off_v.std()**2) / 2)
    return abs(on_v.mean() - off_v.mean()) / (ps + 1e-10)
d_rank = {c: i for i, (c,_) in enumerate(sorted([(c, cohens_d(c)) for c in feature_cols], key=lambda x: -x[1]))}
rf_sel = RandomForestClassifier(100, random_state=42, class_weight='balanced')
rf_sel.fit(StandardScaler().fit_transform(X_all), y_true)
rf_rank = {c: i for i, (c,_) in enumerate(sorted(zip(feature_cols, rf_sel.feature_importances_), key=lambda x: -x[1]))}
combined = sorted(feature_cols, key=lambda c: mi_rank[c] + d_rank[c] + rf_rank[c])
S4_10 = combined[:10]
S4_10_inv = [c for c in combined if c in set(SCALE_INVARIANT)][:10]

ALL_SUBSETS = {
    'S1: Total Power (1)': S1, 'S2: Band Powers (5)': S2,
    'S3a: +Log (10)': S3a, 'S3b: +Band Stats (22)': S3b,
    'S3c: +Ratios+Fracs': S3c, 'S3d: +Shape': S3d,
    'S3e: All Features': S3e,
    'SI: Scale-Invariant': S_INV,
    'SI-r: Ratios+Fracs Only': S_INV_R,
    'S4: Top-10 Guided': S4_10,
    'S4i: Top-10 Inv. Guided': S4_10_inv,
}
for name, feats in ALL_SUBSETS.items():
    print(f'  {name}: {len(feats)} features')

---
## 4. Clustering Methods (100% Coverage)

**Critical:** All methods must predict every sample. DBSCAN noise points are assigned to the nearest cluster centroid.

In [ ]:
METHODS = ['KMeans','GMM','Spectral','Agglomerative','DBSCAN','OneClassSVM','MMC']

def match_labels(yt, yp):
    """Find optimal cluster-to-class mapping. Evaluates on ALL samples."""
    best_f1, best_p, best_r, best_a, best_m = -1, 0, 0, 0, yp.copy()
    for perm in permutations([0, 1]):
        mapping = {old: new for old, new in zip(sorted(np.unique(yp)), perm)}
        mapped = np.array([mapping.get(p, 0) for p in yp])
        f = f1_score(yt, mapped, zero_division=0)
        if f > best_f1:
            best_f1 = f
            best_p = precision_score(yt, mapped, zero_division=0)
            best_r = recall_score(yt, mapped, zero_division=0)
            best_a = accuracy_score(yt, mapped)
            best_m = mapped
    return best_p, best_r, best_f1, best_a, best_m

def run_clustering(X, method, rs=42):
    """All methods return a label for every sample."""
    n = len(X)
    if method == 'KMeans': return KMeans(2, n_init=20, random_state=rs).fit_predict(X)
    elif method == 'GMM': return GaussianMixture(2, n_init=10, random_state=rs, covariance_type='full').fit_predict(X)
    elif method == 'Spectral':
        if X.shape[1] < 2 or n < 4: return KMeans(2, n_init=10, random_state=rs).fit_predict(X)
        return SpectralClustering(2, affinity='rbf', random_state=rs, n_init=10).fit_predict(X)
    elif method == 'Agglomerative': return AgglomerativeClustering(2, linkage='ward').fit_predict(X)
    elif method == 'DBSCAN':
        nn = NearestNeighbors(n_neighbors=min(3, n-1)); nn.fit(X)
        dists, _ = nn.kneighbors(X); eps = np.median(dists[:, -1]) * 1.2
        raw = DBSCAN(eps=eps, min_samples=2).fit_predict(X)
        clusters = set(raw) - {-1}
        if len(clusters) == 0:
            return KMeans(2, n_init=20, random_state=rs).fit_predict(X)
        if len(clusters) == 1:
            other = 1 if 0 in clusters else 0
            raw[raw == -1] = other
            return raw
        centroids = {c: X[raw==c].mean(axis=0) for c in clusters}
        noise_mask = raw == -1
        if noise_mask.any():
            for i in np.where(noise_mask)[0]:
                dists_c = {c: np.linalg.norm(X[i] - centroids[c]) for c in clusters}
                raw[i] = min(dists_c, key=dists_c.get)
        return raw
    elif method == 'OneClassSVM':
        return np.where(OneClassSVM(kernel='rbf', gamma='scale', nu=0.4).fit_predict(X) == 1, 0, 1)
    elif method == 'MMC':
        best = KMeans(2, n_init=10, random_state=rs).fit_predict(X)
        for _ in range(15):
            svc = SVC(kernel='rbf', gamma='scale', C=1.0); svc.fit(X, best)
            new = svc.predict(X)
            if np.array_equal(new, best): break
            best = new
        return best

print(f'Methods: {METHODS}')

---
## 5. Site-Agnostic Evaluation (100% Coverage)

Each method runs **independently per site** with identical hyperparameters. All samples must be predicted.

In [ ]:
site_results = []; site_matched = {}

for sn in sites:
    mk = data['site'] == sn; ys = data.loc[mk, 'running'].values
    for ss_name, feat_list in ALL_SUBSETS.items():
        vf = [c for c in feat_list if c in data.columns]
        Xs = preprocess(data.loc[mk, vf].values)
        for method in METHODS:
            try:
                labels = run_clustering(Xs, method)
                assert len(labels) == len(ys) and (labels >= 0).all()
                prec, rec, f1, acc, matched = match_labels(ys, labels)
                ari = adjusted_rand_score(ys, labels)
                site_results.append({'site': sn, 'subset': ss_name, 'method': method,
                    'Precision': prec, 'Recall': rec, 'F1': f1, 'Accuracy': acc,
                    'ARI': ari, 'n_features': len(vf)})
                site_matched[(sn, ss_name, method)] = matched
            except Exception as e:
                site_results.append({'site': sn, 'subset': ss_name, 'method': method,
                    'Precision': 0, 'Recall': 0, 'F1': 0, 'Accuracy': 0, 'ARI': 0, 'error': str(e)})

site_df = pd.DataFrame(site_results)
print(f'{len(site_df)} per-site experiments (100% coverage)')

agg = site_df.groupby(['subset','method']).agg(
    mean_F1=('F1','mean'), min_F1=('F1','min'), std_F1=('F1','std'),
    mean_P=('Precision','mean'), mean_R=('Recall','mean')).reset_index().sort_values('mean_F1', ascending=False)

print('\nTOP-10 SITE-AGNOSTIC CONFIGS (100% coverage):')
for i, (_, r) in enumerate(agg.head(10).iterrows()):
    print(f'  {i+1}. {r["method"]:<15s} {r["subset"]:<25s} P={r["mean_P"]:.3f} R={r["mean_R"]:.3f} F1={r["mean_F1"]:.3f} (min={r["min_F1"]:.3f})')

---
## 6. Ensemble Methods

In [ ]:
ENSEMBLE_BASE = ['KMeans','GMM','Spectral','Agglomerative','MMC']
ens_results = []

for sn in sites:
    mk = data['site'] == sn; ys = data.loc[mk, 'running'].values
    for ss_name, feat_list in ALL_SUBSETS.items():
        vf = [c for c in feat_list if c in data.columns]
        Xs = preprocess(data.loc[mk, vf].values)
        base_preds = {}; base_f1s = {}
        for method in ENSEMBLE_BASE:
            key = (sn, ss_name, method)
            if key in site_matched:
                base_preds[method] = site_matched[key]
                row = site_df[(site_df['site']==sn)&(site_df['subset']==ss_name)&(site_df['method']==method)]
                base_f1s[method] = row['F1'].values[0] if len(row) > 0 else 0
        if len(base_preds) < 3: continue
        preds_arr = np.column_stack([base_preds[m] for m in base_preds])

        mv = np.array([Counter(row).most_common(1)[0][0] for row in preds_arr])
        p,r,f,a,_ = match_labels(ys, mv)
        ens_results.append({'site':sn,'subset':ss_name,'method':'Ensemble-MV',
            'Precision':p,'Recall':r,'F1':f,'Accuracy':a})

        weights = np.array([base_f1s[m] for m in base_preds])
        weights = weights / (weights.sum() + 1e-10)
        wv = np.zeros(mk.sum())
        for mi, m in enumerate(base_preds): wv += weights[mi] * base_preds[m]
        wv = (wv >= 0.5).astype(int)
        p,r,f,a,_ = match_labels(ys, wv)
        ens_results.append({'site':sn,'subset':ss_name,'method':'Ensemble-WV',
            'Precision':p,'Recall':r,'F1':f,'Accuracy':a})

ens_df = pd.DataFrame(ens_results)
combined_df = pd.concat([site_df, ens_df], ignore_index=True)
ALL_METHODS = METHODS + ['Ensemble-MV', 'Ensemble-WV']

final_agg = combined_df.groupby(['subset','method']).agg(
    mean_F1=('F1','mean'), min_F1=('F1','min'), std_F1=('F1','std'),
    mean_P=('Precision','mean'), mean_R=('Recall','mean')).reset_index().sort_values('mean_F1', ascending=False)

print('TOP-10 (incl. ensembles, 100% coverage):')
for i, (_, r) in enumerate(final_agg.head(10).iterrows()):
    print(f'  {i+1}. {r["method"]:<15s} {r["subset"]:<25s} P={r["mean_P"]:.3f} R={r["mean_R"]:.3f} F1={r["mean_F1"]:.3f} (min={r["min_F1"]:.3f})')

---
## 7. Results & Per-Unit Detail

In [ ]:
print('\n' + '='*100)
print('FULL SITE-AGNOSTIC RESULTS (100% coverage, sorted by mean F1)')
print('='*100)
print(f'{"Method":<15s} {"Subset":<28s} {"Mean P":>7s} {"Mean R":>7s} {"Mean F1":>8s} {"Min F1":>7s} {"Std":>6s}')
print('-'*80)
for _, r in final_agg.iterrows():
    print(f'{r["method"]:<15s} {r["subset"]:<28s} {r["mean_P"]:>7.3f} {r["mean_R"]:>7.3f} {r["mean_F1"]:>8.3f} {r["min_F1"]:>7.3f} {r["std_F1"]:>6.3f}')

In [ ]:
prod = final_agg.iloc[0]
print(f'\nPRODUCTION CANDIDATE: {prod["method"]} + {prod["subset"]}')
print(f'Mean F1={prod["mean_F1"]:.3f}  Min F1={prod["min_F1"]:.3f}\n')

for sn in sites:
    mk = data['site']==sn
    key = (sn, prod['subset'], prod['method'])
    if key in site_matched:
        matched = site_matched[key]
        ys = data.loc[mk,'running'].values
        units = data.loc[mk,'unit'].values
        states = data.loc[mk,'state'].values
        print(f'[{sn}]')
        for u, s, y, p in zip(units, states, ys, matched):
            result = 'OK' if y == p else 'MISS'
            print(f'  {u:<10s} {s.upper():<5s} true={y} pred={int(p)} {result}')
        row = combined_df[(combined_df['site']==sn)&(combined_df['subset']==prod['subset'])&(combined_df['method']==prod['method'])]
        if len(row) > 0:
            r = row.iloc[0]
            print(f'  => P={r["Precision"]:.3f} R={r["Recall"]:.3f} F1={r["F1"]:.3f}\n')

In [ ]:
combined_df.to_csv('_unsupervised_v3_results.csv', index=False)
final_agg.to_csv('_unsupervised_v3_aggregated.csv', index=False)
print('Results exported.')